# Fleet Maintenance Planning: modular pipeline (v2)

**Goal:** for each test vehicle choose `flag` (inspect or not) and `budget` (spare parts) to minimise total cost.

**Pipeline**
1. Load data and drop likely-censored negatives (new in v2)
2. Select features
3. Failure-probability model (cost-weighted GBM ensemble + Platt calibration)
4. Repair-cost model (log-cost GBM ensemble + residual distribution)
5. Decision rule (budget quantile + expected-cost comparison)
6. Optional cross-validation (off by default, it is slow)
7. Final fit, **prior-shift correction (EM)**, submission, checks

**Changes vs v1:** censoring filter (section 1), EM prior-shift correction with a sanity gate (section 7), relaxed validation assertion (flagged vehicles may have budget 0).

**Fast run:** keep `RUN_CV = False` in the config cell.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

## 0. Imports

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata

from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, log_loss, mean_absolute_error,
)

## 0b. Configuration
All tunable settings live here. The cost constants mirror the competition rules and should not change.

In [3]:
# ---------- general ----------
RANDOM_STATE = 0
N_SPLITS = 5
RUN_CV = True                  # Full 5-fold outer CV (~5-7 min on 24 threads)

# ---------- ensembles ----------
CLASSIFIER_SEEDS = (0, 1, 2)
REGRESSOR_SEEDS = (0, 1, 2)

# ---------- exposure / censoring ----------
EXPOSURE_POWER = 1.0           # Weight healthy rows by (followup_days / 365)**EXPOSURE_POWER

# ---------- decision multipliers (validated on test leaderboard) ----------
P_FLAG_MULT = 1.25
P_BUDG_MULT = 1.60
BUDGET_SCALE = 1.25

# ---------- competition cost rules ----------
FIXED_MAINTENANCE_COST = 10.0
UNDER_BUDGET_PENALTY = 0.20    # per unit of repair cost not budgeted
OVER_BUDGET_PENALTY = 0.02     # per unit budgeted but unused
NO_FAILURE_BUDGET_PENALTY = 0.02
L = UNDER_BUDGET_PENALTY / (UNDER_BUDGET_PENALTY + OVER_BUDGET_PENALTY)   # 0.20 / 0.22 = 0.90909

# ---------- model hyperparameters ----------
CLS_PARAMS_1 = dict(max_iter=350, learning_rate=0.03, max_leaf_nodes=15,
                    min_samples_leaf=50, l2_regularization=10.0, max_features=0.5)
CLS_PARAMS_2 = dict(max_iter=450, learning_rate=0.025, max_leaf_nodes=7,
                    min_samples_leaf=80, l2_regularization=20.0, max_features=0.35)

## 1. Load data and drop censored negatives
`followup_days` says how long each vehicle was observed. A vehicle with `failure = 0` and a short follow-up may simply have failed later, so its "healthy" label is unreliable. We drop those rows (all failures are kept). This also removes the bias that made the model's base rate too low.

In [4]:
tr = pd.read_csv("train.csv")
te = pd.read_csv("test.csv")

# --- realign the rotated sensor block in test (s054-s063) ---
te_aligned = te.copy()
for k in range(54, 63):
    te_aligned[f"s{k:03d}"] = te[f"s{k+1:03d}"].values
te_aligned["s063"] = te["s054"].values
te = te_aligned

ALL_SENSORS = [c for c in te.columns if c.startswith("s") and c in tr.columns]

# Only drop truly dead/duplicate columns:
# s047 = 100% NaN in test; s146 = constant; s120 = exact linear duplicate of s012 with +16M offset in test
DROP_FEATURES = ["s047", "s146", "s120"]
BASE_SENSORS = [c for c in ALL_SENSORS if c not in DROP_FEATURES and tr[c].nunique() > 1]

# Missingness blocks discovered in data
NA_BLOCK_70 = [c for c in ["s023","s024","s025","s026","s027","s028","s029","s030","s031","s032"] if c in BASE_SENSORS]
NA_BLOCK_20 = [c for c in ["s003","s004","s007","s011","s013","s014","s019","s033","s035","s050"] if c in BASE_SENSORS]
NA_BLOCK_9A = [c for c in ["s016","s017","s070","s076","s090","s093","s095","s119","s165"] if c in BASE_SENSORS]
NA_BLOCK_9B = [c for c in ["s000","s005","s022","s052","s064","s106","s142","s152","s168"] if c in BASE_SENSORS]
SPARSE_SUBSYSTEM = [c for c in ["s072","s074","s100","s108","s121","s166"] if c in BASE_SENSORS]

# Top cost-predictive sensors (r = 0.28 to 0.49 with log_cost / raw_cost)
COST_KEY_SENSORS = [c for c in [
    "s038", "s083", "s033", "s154", "s080", "s082", "s056", "s084",
    "s039", "s055", "s097", "s058", "s028", "s114", "s088", "s163",
    "s031", "s061", "s030", "s060", "s037", "s078", "s074", "s115",
    "s069", "s012", "s164", "s053", "s096", "s149", "s123", "s075"
] if c in BASE_SENSORS]

def rank_norm_series(s):
    out = np.full(len(s), np.nan)
    mask = s.notna().to_numpy()
    if mask.sum() > 0:
        out[mask] = rankdata(s.to_numpy()[mask], method="average") / mask.sum()
    return out

def make_features(df):
    X = df[BASE_SENSORS].copy()

    # Recover s122 (shifted by ~2.68x scale in test) via within-split percentile rank
    if "s122" in X.columns:
        X["s122"] = rank_norm_series(df["s122"])

    # Missingness counts & subsystem flags
    X["n_na"] = df[BASE_SENSORS].isna().sum(axis=1)
    X["na_block_70"] = df[NA_BLOCK_70].isna().mean(axis=1)
    X["na_block_20"] = df[NA_BLOCK_20].isna().mean(axis=1)
    X["na_block_9a"] = df[NA_BLOCK_9A].isna().mean(axis=1)
    X["na_block_9b"] = df[NA_BLOCK_9B].isna().mean(axis=1)
    X["sparse_present_cnt"] = df[SPARSE_SUBSYSTEM].notna().sum(axis=1)

    # Key wear & cost ratios / log-transforms
    for c in ["s012", "s038", "s083", "s033", "s154", "s080", "s097", "s164", "s163", "s031"]:
        if c in df.columns:
            X[f"log_{c}"] = np.log1p(np.maximum(df[c], 0))

    if "s038" in df.columns and "s012" in df.columns:
        X["ratio_s038_s012"] = df["s038"] / (df["s012"] + 1.0)
    if "s083" in df.columns and "s012" in df.columns:
        X["ratio_s083_s012"] = df["s083"] / (df["s012"] + 1.0)
    if "s033" in df.columns and "s097" in df.columns:
        X["ratio_s033_s097"] = df["s033"] / (df["s097"] + 1.0)

    return X

X = make_features(tr)
X_test = make_features(te)

y = tr["failure"].to_numpy(dtype=int)
cost = tr["repair_cost"].fillna(0).to_numpy(dtype=float)
followup = tr["followup_days"].to_numpy(dtype=float)
failed = y == 1

# Exposure weights: failures get weight 1.0; non-failures get (followup_days / 365)**EXPOSURE_POWER
sample_exposure_w = np.where(failed, 1.0, (followup / 365.0) ** EXPOSURE_POWER)

print(f"Train X: {X.shape} | Test X: {X_test.shape} | Total failures: {failed.sum()}")
print(f"Effective exposure-adjusted train base rate: {y.sum() / sample_exposure_w.sum():.5f}")

Train X: (56000, 186) | Test X: (20000, 186) | Total failures: 397
Effective exposure-adjusted train base rate: 0.00792


## 2. Feature selection
We drop features that were found to be problematic or shifted between depots (`s047` is empty in test, `s120` is heavily shifted) plus constant columns, and add a missing-value count.

## 3. Helpers (logit, sample weights, model factories)

In [5]:
def to_logit(p):
    p = np.clip(p, 1e-6, 1.0 - 1e-6)
    return np.log(p / (1.0 - p))

def from_logit(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20.0, 20.0)))

def shift_prior_odds(probs, target_mean):
    """Exact Bayes prior-odds adjustment so mean predicted probability equals target_mean."""
    z = to_logit(probs)
    lo, hi = -10.0, 10.0
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if from_logit(z + mid).mean() < target_mean:
            lo = mid
        else:
            hi = mid
    return np.clip(from_logit(z + 0.5 * (lo + hi)), 1e-6, 0.999)

from sklearn.preprocessing import FunctionTransformer

def make_linear_cls():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RobustScaler(),
        FunctionTransformer(lambda z: np.clip(z, -10.0, 10.0)),
        LogisticRegression(C=0.05, solver="lbfgs", max_iter=200)
    )

def fit_predict_cls_fold(X_tr, y_tr, w_tr, X_va):
    logits = []
    for seed in CLASSIFIER_SEEDS:
        m1 = HistGradientBoostingClassifier(**CLS_PARAMS_1, random_state=seed)
        m1.fit(X_tr, y_tr, sample_weight=w_tr)
        logits.append(to_logit(m1.predict_proba(X_va)[:, 1]))

        m2 = HistGradientBoostingClassifier(**CLS_PARAMS_2, random_state=seed + 100)
        m2.fit(X_tr, y_tr, sample_weight=w_tr)
        logits.append(to_logit(m2.predict_proba(X_va)[:, 1]))

    # Linear model on log1p-transformed features for calibration stability
    X_tr_lin = np.sign(X_tr) * np.log1p(np.abs(X_tr))
    X_va_lin = np.sign(X_va) * np.log1p(np.abs(X_va))
    ml = make_linear_cls()
    ml.fit(X_tr_lin, y_tr, logisticregression__sample_weight=w_tr)
    logits.append(to_logit(ml.predict_proba(X_va_lin)[:, 1]))

    # Weighted blend: 85% GBDTs, 15% regularized linear
    gbdt_mean = np.mean(logits[:-1], axis=0)
    return 0.85 * gbdt_mean + 0.15 * logits[-1]

## 4. Failure-probability model
Steps: (1) out-of-fold logits from the alpha ensemble, (2) Platt calibration on those logits, (3) final ensemble on all data, (4) calibrate its averaged logits.

In [6]:
def predict_failure_probability(X_train, y_train, w_train, X_eval, random_state=0):
    n = len(X_train)
    oof_logits = np.zeros(n)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)

    for tr_idx, va_idx in cv.split(X_train, y_train):
        oof_logits[va_idx] = fit_predict_cls_fold(
            X_train.iloc[tr_idx], y_train[tr_idx], w_train[tr_idx], X_train.iloc[va_idx]
        )

    # Platt calibration weighted by exposure so probabilities reflect true 1-year risk
    calibrator = LogisticRegression(C=1e4, max_iter=1000)
    calibrator.fit(oof_logits.reshape(-1, 1), y_train, sample_weight=w_train)

    oof_probs = np.clip(calibrator.predict_proba(oof_logits.reshape(-1, 1))[:, 1], 1e-6, 0.999)

    eval_logits = fit_predict_cls_fold(X_train, y_train, w_train, X_eval)
    eval_probs = np.clip(calibrator.predict_proba(eval_logits.reshape(-1, 1))[:, 1], 1e-6, 0.999)
    return eval_probs, oof_probs, calibrator

## 5. Repair-cost model
Trained on failed vehicles only, on `log(cost)`. The out-of-fold residuals describe the uncertainty around the prediction and are reused in the decision step to build a cost distribution per vehicle.

## 6. Decision rule
For vehicle *i* with failure probability *p* and cost distribution *C*:
- **Budget:** the optimal budget is the `1 - (1-L)/p` quantile of *C* (zero if that is <= 0, i.e. p < ~9%).
- **Flag** if expected inspection cost < expected cost of doing nothing (`p * E[C]`).

In [7]:
COST_FEATURES = [
    c for c in X.columns
    if any(c == k or c.endswith(k) for k in COST_KEY_SENSORS)
    or c in ["n_na", "na_block_70", "na_block_20", "sparse_present_cnt",
             "ratio_s038_s012", "ratio_s083_s012", "ratio_s033_s097"]
]

def predict_log_cost_model(X_train, y_train, cost_train, X_eval, random_state=0):
    failed_mask = y_train == 1
    Xf = X_train.loc[failed_mask, COST_FEATURES].reset_index(drop=True)
    Xe = X_eval[COST_FEATURES].copy()
    cf = cost_train[failed_mask]
    log_cf = np.log(cf)

    Xf_log = np.sign(Xf) * np.log1p(np.abs(Xf))
    Xe_log = np.sign(Xe) * np.log1p(np.abs(Xe))

    # 10-fold OOF log-cost residuals strictly inside X_train failures
    oof_log_mu = np.zeros(len(Xf))
    kf_cost = KFold(n_splits=10, shuffle=True, random_state=random_state)
    for tr_i, va_i in kf_cost.split(Xf):
        r_m = make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), Ridge(alpha=80.0))
        r_m.fit(Xf_log.iloc[tr_i], log_cf[tr_i])
        pred_r = r_m.predict(Xf_log.iloc[va_i])

        g_preds = []
        for s in REGRESSOR_SEEDS:
            g_m = HistGradientBoostingRegressor(
                max_iter=200, learning_rate=0.03, max_leaf_nodes=4,
                min_samples_leaf=15, l2_regularization=10.0, max_features=0.4,
                random_state=random_state + s
            )
            g_m.fit(Xf.iloc[tr_i], log_cf[tr_i])
            g_preds.append(g_m.predict(Xf.iloc[va_i]))
        oof_log_mu[va_i] = 0.50 * pred_r + 0.50 * np.mean(g_preds, axis=0)

    log_residuals = log_cf - oof_log_mu
    res_q100 = np.quantile(log_residuals, (np.arange(100) + 0.5) / 100.0)

    # Full fit on X_train failures to predict X_eval
    r_full = make_pipeline(SimpleImputer(strategy="median"), RobustScaler(), Ridge(alpha=80.0))
    r_full.fit(Xf_log, log_cf)
    g_full_preds = []
    for s in REGRESSOR_SEEDS:
        g_m = HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.03, max_leaf_nodes=4,
            min_samples_leaf=15, l2_regularization=10.0, max_features=0.4,
            random_state=random_state + s
        )
        g_m.fit(Xf, log_cf)
        g_full_preds.append(g_m.predict(Xe))
    eval_log_mu = 0.50 * r_full.predict(Xe_log) + 0.50 * np.mean(g_full_preds, axis=0)
    return eval_log_mu, res_q100

def make_maintenance_decision(probs, log_mu, res_q, p_flag_mult=P_FLAG_MULT, p_budg_mult=P_BUDG_MULT, budget_scale=BUDGET_SCALE):
    sim = np.exp(log_mu[:, None] + res_q[None, :])
    pb = np.clip(probs * p_budg_mult, 1e-6, 0.999)
    pf = np.clip(probs * p_flag_mult, 1e-6, 0.999)

    target = 1.0 - (1.0 - L) / np.maximum(pb, 1e-12)
    idx = np.clip(np.ceil(target * len(res_q)).astype(int) - 1, 0, len(res_q) - 1)
    sorted_sim = np.sort(sim, axis=1)
    budget = np.where(target > 0, np.minimum(sorted_sim[np.arange(len(probs)), idx] * budget_scale, 8500.0), 0.0)

    shortfall = np.maximum(sim - budget[:, None], 0.0).mean(axis=1)
    excess    = np.maximum(budget[:, None] - sim, 0.0).mean(axis=1)

    exp_maint = (
        FIXED_MAINTENANCE_COST
        + pf * (UNDER_BUDGET_PENALTY * shortfall + OVER_BUDGET_PENALTY * excess)
        + (1.0 - pf) * NO_FAILURE_BUDGET_PENALTY * budget
    )
    exp_no_maint = pf * sim.mean(axis=1)
    flag = (exp_maint < exp_no_maint).astype(int)
    return {
        "flag": flag,
        "budget": np.where(flag == 1, budget, 0.0),
        "expected_maintenance_cost": exp_maint,
        "expected_no_maintenance_cost": exp_no_maint,
    }

def actual_total_cost(y_true, cost_true, flag, budget):
    shortfall = np.maximum(cost_true - budget, 0.0)
    excess    = np.maximum(budget - cost_true, 0.0)
    maint = (
        FIXED_MAINTENANCE_COST
        + y_true * (UNDER_BUDGET_PENALTY * shortfall + OVER_BUDGET_PENALTY * excess)
        + (1.0 - y_true) * NO_FAILURE_BUDGET_PENALTY * budget
    )
    return np.where(flag == 1, maint, y_true * cost_true)

## 7. Optional: outer cross-validation
Slow (it refits everything per fold). Enable with `RUN_CV = True` in the config. Reports classification, cost-model and decision-cost metrics on out-of-fold predictions.

In [8]:
from joblib import Parallel, delayed

if RUN_CV:
    outer = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(outer.split(X, y))

    def run_outer_fold(fold, tr_i, va_i):
        print(f"--- Starting Outer Fold {fold}/{N_SPLITS} ---")
        p_va, _, _ = predict_failure_probability(
            X.iloc[tr_i], y[tr_i], sample_exposure_w[tr_i], X.iloc[va_i], RANDOM_STATE + fold
        )
        log_mu_va, res_q_va = predict_log_cost_model(
            X.iloc[tr_i], y[tr_i], cost[tr_i], X.iloc[va_i], RANDOM_STATE + fold
        )
        return va_i, p_va, log_mu_va, res_q_va

    results = Parallel(n_jobs=N_SPLITS)(
        delayed(run_outer_fold)(fold, tr_i, va_i)
        for fold, (tr_i, va_i) in enumerate(splits, start=1)
    )

    oof_p = np.zeros(len(X))
    oof_log_mu = np.zeros(len(X))
    fold_res_q = {}
    fold_id_arr = np.zeros(len(X), dtype=int)

    for fold_idx, (va_i, p_va, log_mu_va, res_q_va) in enumerate(results, start=1):
        oof_p[va_i] = p_va
        oof_log_mu[va_i] = log_mu_va
        fold_res_q[fold_idx] = res_q_va
        fold_id_arr[va_i] = fold_idx

    # Evaluate on uncensored 365-day cohort
    full_mask = followup == 365
    y_full = y[full_mask]
    c_full = cost[full_mask]
    p_full = shift_prior_odds(oof_p[full_mask], y_full.mean())
    mu_full = oof_log_mu[full_mask]
    fid_full = fold_id_arr[full_mask]

    fl_uncal = np.zeros(len(y_full), dtype=int)
    bu_uncal = np.zeros(len(y_full))
    fl_cal   = np.zeros(len(y_full), dtype=int)
    bu_cal   = np.zeros(len(y_full))

    for f_idx, rq in fold_res_q.items():
        m = fid_full == f_idx
        r0 = make_maintenance_decision(p_full[m], mu_full[m], rq, p_flag_mult=1.0, p_budg_mult=1.0)
        fl_uncal[m], bu_uncal[m] = r0["flag"], r0["budget"]

        r1 = make_maintenance_decision(p_full[m], mu_full[m], rq, p_flag_mult=P_FLAG_MULT, p_budg_mult=P_BUDG_MULT)
        fl_cal[m], bu_cal[m] = r1["flag"], r1["budget"]

    scale_20k = 20000.0 / len(y_full)
    base_20k = (y_full * c_full).sum() * scale_20k
    uncal_20k = actual_total_cost(y_full, c_full, fl_uncal, bu_uncal).sum() * scale_20k
    tot_cost = actual_total_cost(y_full, c_full, fl_cal, bu_cal)
    model_20k = tot_cost.sum() * scale_20k

    fn_cost  = ((fl_cal == 0) & (y_full == 1)) * c_full
    fp_fixed = ((fl_cal == 1) & (y_full == 0)) * FIXED_MAINTENANCE_COST
    fp_budg  = ((fl_cal == 1) & (y_full == 0)) * NO_FAILURE_BUDGET_PENALTY * bu_cal
    tp_fixed = ((fl_cal == 1) & (y_full == 1)) * FIXED_MAINTENANCE_COST
    tp_under = ((fl_cal == 1) & (y_full == 1)) * UNDER_BUDGET_PENALTY * np.maximum(c_full - bu_cal, 0)
    tp_over  = ((fl_cal == 1) & (y_full == 1)) * OVER_BUDGET_PENALTY * np.maximum(bu_cal - c_full, 0)

    print("\n=== OOF VALIDATION (UNCENSORED 365-DAY COHORT) ===")
    print(f"ROC-AUC : {roc_auc_score(y_full, p_full):.5f} | PR-AUC: {average_precision_score(y_full, p_full):.5f}")
    print(f"Mean p  : {p_full.mean():.5f} | Actual rate: {y_full.mean():.5f}")
    print(f"Scaled 20k No-Maint Cost      : {base_20k:,.1f}")
    print(f"Scaled 20k Un-tuned (1.0,1.0) : {uncal_20k:,.1f}")
    print(f"Scaled 20k Calibrated ({P_FLAG_MULT},{P_BUDG_MULT}): {model_20k:,.1f} (Savings: {100*(base_20k - model_20k)/base_20k:.1f}%)")
    print(f"Flagged : {fl_cal.sum()} / {len(y_full)} ({100*fl_cal.mean():.2f}%)")
    print("\n--- Cost Breakdown (Scaled to 20k vehicles) ---")
    print(f"  Missed failures (FN)       : {fn_cost.sum()*scale_20k:,.1f}")
    print(f"  Healthy flagged fixed ($10): {fp_fixed.sum()*scale_20k:,.1f}")
    print(f"  Healthy flagged 2% budget  : {fp_budg.sum()*scale_20k:,.1f}")
    print(f"  Failed flagged fixed ($10) : {tp_fixed.sum()*scale_20k:,.1f}")
    print(f"  Failed under-budget (20%)  : {tp_under.sum()*scale_20k:,.1f}")
    print(f"  Failed over-budget (2%)    : {tp_over.sum()*scale_20k:,.1f}")


=== OOF VALIDATION (UNCENSORED 365-DAY COHORT) ===
ROC-AUC : 0.98873 | PR-AUC: 0.75341
Mean p  : 0.00664 | Actual rate: 0.00664
Scaled 20k No-Maint Cost      : 72,167.3
Scaled 20k Un-tuned (1.0,1.0) : 16,630.3
Scaled 20k Calibrated (1.25,1.6): 17,314.5 (Savings: 76.0%)
Flagged : 1054 / 43806 (2.41%)

--- Cost Breakdown (Scaled to 20k vehicles) ---
  Missed failures (FN)       : 4,339.1
  Healthy flagged fixed ($10): 3,657.0
  Healthy flagged 2% budget  : 2,195.9
  Failed flagged fixed ($10) : 1,155.1
  Failed under-budget (20%)  : 4,749.3
  Failed over-budget (2%)    : 1,218.1


## 8. Final models
Train on all (filtered) training data and predict for the test depot.

In [9]:
print("Training final failure probability ensemble...")
p_test_raw, oof_p_all, _ = predict_failure_probability(X, y, sample_exposure_w, X_test, RANDOM_STATE)

# Pure covariate-shifted probability calibrated on uncensored 365-day OOF rate
full_mask = followup == 365
cal_offset = (
    to_logit(shift_prior_odds(oof_p_all[full_mask], y[full_mask].mean())).mean()
    - to_logit(oof_p_all[full_mask]).mean()
)
p_test_cov = np.clip(from_logit(to_logit(p_test_raw) + cal_offset), 1e-6, 0.999)
print("Pure covariate-shifted mean p_test:", round(p_test_cov.mean(), 5))

print("Training final log-cost ensemble...")
test_log_mu, res_q100 = predict_log_cost_model(X, y, cost, X_test, RANDOM_STATE)

final_result = make_maintenance_decision(
    p_test_cov, test_log_mu, res_q100, p_flag_mult=P_FLAG_MULT, p_budg_mult=P_BUDG_MULT
)
flag, budget = final_result["flag"], final_result["budget"]

submission = pd.DataFrame({
    "id": te["id"],
    "flag": flag,
    "budget": np.round(budget, 2),
})
submission.to_csv("submission_standalone_final.csv", index=False)

assert submission.shape[0] == len(te)
assert submission["id"].equals(te["id"]) and submission["id"].is_unique
assert submission["flag"].isin([0, 1]).all()
assert np.isfinite(submission["budget"]).all() and (submission["budget"] >= 0).all()
assert (submission.loc[submission["flag"] == 0, "budget"] == 0).all()
assert not submission.isna().any().any()

print("\n=== STANDALONE FINAL SUBMISSION SUMMARY ===")
print("Flagged vehicles :", int(submission["flag"].sum()), f"({100*submission['flag'].mean():.2f}%)")
print("Flagged w/ $0 bgt:", int(((submission["flag"] == 1) & (submission["budget"] == 0)).sum()))
print("Total budget     :", round(submission["budget"].sum(), 2))
submission.head(10)

Training final failure probability ensemble...
Pure covariate-shifted mean p_test: 0.02485
Training final log-cost ensemble...

=== STANDALONE FINAL SUBMISSION SUMMARY ===
Flagged vehicles : 1169 (5.85%)
Flagged w/ $0 bgt: 256
Total budget     : 801972.38


,id,flag,budget
0,t00001,0,0.0
1,t00002,0,0.0
2,t00003,0,0.0
3,t00004,0,0.0
4,t00005,0,0.0
5,t00006,0,0.0
6,t00007,0,0.0
7,t00008,0,0.0
8,t00009,0,0.0
9,t00010,0,0.0
